# Parameters, State, and Memory

Almost everything we do to a model other than calling it operates on its
*state*: the optimizer updates it, a checkpoint serializes it, device
placement moves it, fine-tuning trains part of it, and the answer to "will
this model fit on my GPU?" is a few lines of arithmetic over it. So far that
state has been handled for us; the layers created the tensors and the
training loop updated them. This section opens the box: how to reach any tensor in a model,
which tensors are trained and which merely travel with the model, what they
all cost in bytes, and how to share or freeze them.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf

## Accessing Parameters

Our specimen is the residual MLP of that section,
redefined here so this section stands on its own: an input layer, a stack of
residual blocks, and an output head.

In [2]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, d):
        super().__init__()
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(d), tf.keras.layers.ReLU(),
            tf.keras.layers.Dense(d)])

    def call(self, X):
        return X + self.body(X)

tf.keras.utils.set_random_seed(42)
net = tf.keras.Sequential([tf.keras.layers.Dense(64), ResidualBlock(64),
                           ResidualBlock(64), tf.keras.layers.Dense(10)])
X = tf.random.normal((2, 20))
net(X).shape

TensorShape([2, 10])

A model built from layers is a tree, and its parameters are the leaves. To
reach one leaf, walk the tree: `net.layers` lists a model's children,
attribute access selects within a layer. `net.layers[3]` is the output layer;
its bias is a `Variable` the layer created through `add_weight`, announcing
itself to Keras as trainable.

In [3]:
type(net.layers[3].bias), net.layers[3].bias.shape

(keras.src.backend.Variable, TensorShape([10]))

The same path syntax reaches arbitrarily deep. The first linear layer inside
the first residual block sits three levels down:

In [4]:
net.layers[1].body.layers[0].kernel.shape

TensorShape([64, 64])

Gradients are not stored on variables. A `tf.GradientTape` records the
forward pass, and its `gradient` method returns the gradients as a *second*
list, parallel to the variables you differentiate with respect to:

In [5]:
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(net(X))
grads = tape.gradient(loss, net.trainable_weights)
len(grads) == len(net.trainable_weights)

True

Reaching parameters one path at a time is right for debugging a single layer.
The optimizer, weight decay, and checkpointing instead need *every* leaf, and
`net.weights` provides exactly that: a traversal of the whole tree, one
variable per leaf, each carrying its path as its name.

In [6]:
[(v.path, v.shape) for v in net.weights]

[('sequential_2/dense/kernel', TensorShape([20, 64])),
 ('sequential_2/dense/bias', TensorShape([64])),
 ('sequential_2/residual_block/sequential/dense_1/kernel',
  TensorShape([64, 64])),
 ('sequential_2/residual_block/sequential/dense_1/bias', TensorShape([64])),
 ('sequential_2/residual_block/sequential/dense_2/kernel',
  TensorShape([64, 64])),
 ('sequential_2/residual_block/sequential/dense_2/bias', TensorShape([64])),
 ('sequential_2/residual_block_1/sequential_1/dense_3/kernel',
  TensorShape([64, 64])),
 ('sequential_2/residual_block_1/sequential_1/dense_3/bias',
  TensorShape([64])),
 ('sequential_2/residual_block_1/sequential_1/dense_4/kernel',
  TensorShape([64, 64])),
 ('sequential_2/residual_block_1/sequential_1/dense_4/bias',
  TensorShape([64])),
 ('sequential_2/dense_5/kernel', TensorShape([64, 10])),
 ('sequential_2/dense_5/bias', TensorShape([10]))]

Read one of the paths closely.
`sequential_2/residual_block/sequential/dense_1/kernel` means: inside the
outer `Sequential`, its child `residual_block` (the first residual block),
that block's `body` (a `Sequential` of its own), the first `Dense` inside it,
and finally the leaf `kernel`. The segments are layer *names*, assigned at
construction: a default derived from the class, plus a counter that keeps
names unique across the program (this is the third `Sequential` created, hence
`sequential_2`). The paths expose ownership for inspection, but default names
depend on construction history and are not an external identifier. Name layers
explicitly when a tool must match paths across separately written programs;
Keras checkpoints otherwise restore through the saved object topology and
weight structure (that section). Keras splits the same list by
trainability, and so far the split is trivial:

In [7]:
[v.path for v in net.weights] == [v.path for v in net.trainable_weights]

True

One tree, one naming scheme, and every consumer, whether optimizer,
checkpoint, or debugger, walks it. The equality above holds for this model
because all of its state happens to be trainable. That is not always so.

## Parameters and Buffers

Some tensors must persist inside a model and follow it from device to device,
yet should never receive a gradient. The canonical example is batch
normalization [@Ioffe.Szegedy.2015]: each layer maintains a running mean
and variance of its inputs, updated during the forward pass and used at
prediction time. Those statistics must be saved with the model and must move
to the GPU with it, but the optimizer has no business touching them. A
precomputed positional table may be treated the same way. Other non-trained
state has a different lifetime: a language model's key--value cache belongs to
one generation request, so it follows the computation device but should not be
stored in a model checkpoint. Causal masks are often recomputed or registered
as non-persistent state for the same reason.

Keras calls every registered tensor a *weight* and distinguishes by a
per-variable flag: `add_weight(trainable=False)` creates a variable that is
saved with the model but never handed to the optimizer, which is how
`BatchNormalization` stores its running statistics. The rule of thumb: make
it a trainable weight if the optimizer should update it, a non-trainable
weight if it must persist in checkpoints, and an explicit runtime input or
cache if its lifetime is one request. Plain Python attributes hold
reconstructible architecture data. Here is a layer that standardizes its inputs with
statistics computed once, ahead of time, from a reference sample:

In [8]:
class Whitener(tf.keras.layers.Layer):
    def __init__(self, mean, std):
        super().__init__()
        self.mean = self.add_weight(
            shape=mean.shape, trainable=False, name='mean',
            initializer=tf.keras.initializers.Constant(mean))
        self.std = self.add_weight(
            shape=std.shape, trainable=False, name='std',
            initializer=tf.keras.initializers.Constant(std))
        self.out = tf.keras.layers.Dense(2)

    def call(self, X):
        return self.out((X - self.mean) / self.std)

sample = tf.random.normal((100, 4)) * tf.range(1., 5.)
whiten = Whitener(tf.reduce_mean(sample, 0), tf.math.reduce_std(sample, 0))
_ = whiten(sample[:2])   # build the Dense layer
[v.path for v in whiten.weights]

['whitener/mean',
 'whitener/std',
 'whitener/dense_6/kernel',
 'whitener/dense_6/bias']

All four variables are weights, so all four are checkpointed. The flag is
what keeps the optimizer away from the statistics: a training step
differentiates and updates `trainable_weights`, and the two constants sit in
the other half of the split:

In [9]:
([v.path for v in whiten.trainable_weights],
 [v.path for v in whiten.non_trainable_weights])

(['whitener/dense_6/kernel', 'whitener/dense_6/bias'],
 ['whitener/mean', 'whitener/std'])

Registration through `add_weight` is what makes the layer aware of a tensor.
A tensor stored as a plain attribute is invisible to it: not counted, not
saved, and absent from every traversal. Attach one and the weights list does
not grow:

In [10]:
whiten.note = tf.zeros(4)   # plain attribute: invisible to the layer
[v.path.split('/')[-1] for v in whiten.weights]

['mean', 'std', 'kernel', 'bias']

Device placement works differently here: a TensorFlow variable's device is
decided when the variable is *created*, not by a later move. With a GPU
visible, Keras creates variables on it by default, non-trainable weights
included, and a `tf.device` scope overrides the choice. There is no `.to()`
to forget, so the classic left-behind-tensor bug cannot arise; the price is
that relocating existing state means rebuilding it. Each variable reports its
placement:

In [11]:
whiten.mean.value.device   # decided when the variable was created

'/job:localhost/replica:0/task:0/device:GPU:0'

## Counting Parameters, Counting Bytes

Before any training job comes the question of whether the model fits in
memory, and the answer is arithmetic you can do on a napkin. Counting
parameters is one line. Counting *bytes* requires remembering everything that
training keeps per parameter: the weight itself, its gradient, and the
optimizer's state. Adam maintains two running moments per parameter, so in
fp32 the ledger reads:

| Training state       | Precision | Bytes per parameter |
|----------------------|-----------|---------------------|
| Weights              | fp32      | 4                   |
| Gradients            | fp32      | 4                   |
| Adam first moment    | fp32      | 4                   |
| Adam second moment   | fp32      | 4                   |
| Total                |           | 16                  |

In [12]:
n = net.count_params()
weights = sum(int(tf.size(v)) * tf.as_dtype(v.dtype).size
              for v in net.weights)   # fp32: 4 bytes
grads, adam_state = weights, 2 * weights
print(f'{n} parameters: {(weights + grads + adam_state) / 2**20:.2f} MiB '
      'for weights + gradients + Adam state')

18634 parameters: 0.28 MiB for weights + gradients + Adam state


The optimizer state is real memory, allocated tensor by tensor. After a
single training step, Adam's two moments together hold exactly two extra
copies of the model:

In [13]:
adam = tf.keras.optimizers.Adam()
adam.build(net.trainable_weights)   # Keras allocates the moments here, up front
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(net(X))
adam.apply_gradients(zip(tape.gradient(loss, net.trainable_weights),
                         net.trainable_weights))
moments = sum(int(tf.size(v)) for v in adam.variables
              if v.ndim > 0)   # ndim 0 excludes the step count and learning rate
moments == 2 * n

True

For our little network the total is a third of a megabyte, which is why none
of this mattered until now. Scale the same arithmetic to a 1-billion-parameter
model and the standard fp32 Adam ledger gives a batch-independent floor: 4 GB
for the weights alone and 16 GB for weights, gradients, and optimizer state,
before storing a single activation (that section). Activations depend on
batch size, sequence length, and architecture; they can exceed parameter state
and are treated in that section.

Large models often train in mixed precision
[@Micikevicius.Narang.Alben.ea.2018], but storage policies differ. One
implementation convention counts 18 bytes per parameter (fp32 master weights,
an fp16 working copy, fp32 gradients, and two fp32 moments); the ZeRO paper
counts 16 by keeping gradients in fp16
[@Rajbhandari.Rasley.Ruwase.ea.2020]. Other bf16 and AMP stacks keep no
separate working copy or choose another dtype for optimizer state. State the
dtypes before quoting a multiplier. In the two conventions above, the fp32
Adam moments alone cost 8 bytes per parameter.


![Bytes per parameter under fp32 Adam training (top) versus a mixed-precision, ZeRO-style convention (bottom), drawn to the same width per byte. The two ledgers disagree on weights and gradients but agree, byte for byte, on the Adam moments m and v.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-memory-ledger.svg)

the figure lays the two conventions side by side, byte for
byte: the moments (right half of each bar) line up exactly, and the only
disagreement is how the weights and gradients themselves are stored.

## Tied Parameters

One tensor can serve several roles in a model. The standard example is the
two ends of a language model. The input embedding is a $|V| \times d$ table
mapping each of $|V|$ tokens to a $d$-dimensional vector; the output
projection maps a $d$-dimensional hidden state to $|V|$ logits, and its weight
matrix has the same shape and an analogous meaning: one vector per token.
*Tying* the two, using a single tensor for both roles, saves $|V| \times d$
parameters. The cited studies also found lower language-model perplexity in
their experimental settings
[@Press.Wolf.2017; @Inan.Khosravi.Socher.2017]. The savings are large: in
GPT-2 [@Radford.Wu.Child.ea.2019] the shared $50257 \times 768$ matrix is
about 38.6 million of the model's 124 million parameters, roughly 31%.

![One weight matrix W, two call sites: the embedding looks up a row by token id, the output head multiplies by its transpose to produce logits. Tying means both point at the same tensor, so gradients from both uses sum into one place.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-weight-tying.svg)

the figure sketches the picture behind the two call sites.

A miniature version shows the mechanics. Keras has no tying flag, so we say
it in code: a small head layer that creates no variables of its own and
instead multiplies by the transpose of the embedding table it is handed. One
variable, two call sites:

In [14]:
class TiedLMHead(tf.keras.layers.Layer):
    def __init__(self, emb):
        super().__init__()
        self.emb = emb   # reuse the embedding layer; create nothing

    def call(self, h):
        return tf.matmul(h, self.emb.embeddings, transpose_b=True)

class TinyLM(tf.keras.Model):
    def __init__(self, vocab_size, d, tied=True):
        super().__init__()
        self.emb = tf.keras.layers.Embedding(vocab_size, d)
        self.body = tf.keras.layers.Dense(d)
        self.head = (TiedLMHead(self.emb) if tied else
                     tf.keras.layers.Dense(vocab_size, use_bias=False))

    def call(self, tokens):
        return self.head(tf.nn.relu(self.body(self.emb(tokens))))

lm = TinyLM(vocab_size=100, d=16)
tokens = tf.random.uniform((2, 8), 0, 100, dtype=tf.int32)
_ = lm(tokens)   # build
[(v.path, tuple(v.shape)) for v in lm.weights]

[('tiny_lm/embedding/embeddings', (100, 16)),
 ('tiny_lm/dense_7/kernel', (16, 16)),
 ('tiny_lm/dense_7/bias', (16,))]

Because the head created no variable, there is no aliasing bookkeeping to get
right: the traversal reports the table once and no head matrix exists. The
parameter count reflects the $|V| \times d$ saving, and the optimizer updates
the table once per step:

In [15]:
untied = TinyLM(vocab_size=100, d=16, tied=False)
_ = untied(tokens)
(lm.count_params(), untied.count_params())

(1872, 3472)

A checkpoint of the tied model stores the table once, because the head owns
nothing to store. The flip side is that tied and untied checkpoints differ in
structure: the untied model carries one extra kernel, so loading one into the
other is a mismatch to resolve explicitly, not a silent aliasing decision.

What about gradients? During backpropagation each use of the tensor
contributes a gradient, and the contributions accumulate into the gradient of
the single shared tensor. We can verify this against the untied twin: load
the tied model's values into it, run the same backward pass through both, and
the tied gradient equals the *sum* of the untied model's two gradients.

In [16]:
untied.set_weights(lm.get_weights()   # tied values, head kernel materialized
                   + [tf.transpose(lm.emb.embeddings)])
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(lm(tokens))
g_tied = tape.gradient(loss, lm.emb.embeddings)
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(untied(tokens))
g_emb, g_head = tape.gradient(loss, [untied.emb.embeddings, untied.head.kernel])
g_emb = tf.convert_to_tensor(g_emb)   # densify the embedding-lookup gradient
bool(tf.reduce_max(tf.abs(g_tied - (g_emb + tf.transpose(g_head)))) < 1e-5)

True

## Freezing Parameters

Every fine-tuning recipe (that section) rests on one primitive,
and in Keras it sits on the layer: setting `layer.trainable = False` moves
the layer's variables from `trainable_weights` to `non_trainable_weights`,
the split a training step consults. (One caveat for the `compile`-and-`fit`
workflow: Keras documents that a change to `trainable` takes effect at the
next `compile`. The manual steps below read the split live.) The typical
pattern freezes a pretrained backbone and trains only a new head. On a fresh
copy of our residual MLP, freezing everything but the output layer leaves 650
of 18,634 parameters trainable:

In [17]:
finetune = tf.keras.Sequential([tf.keras.layers.Dense(64), ResidualBlock(64),
                                ResidualBlock(64), tf.keras.layers.Dense(10)])
_ = finetune(X)   # build
for layer in finetune.layers[:-1]:
    layer.trainable = False

(sum(int(tf.size(v)) for v in finetune.trainable_weights),
 finetune.count_params())

(650, 18634)

The tape differentiates with respect to the variables you hand it, and
`trainable_weights` is now exactly the head. One gradient step then moves the
head and nothing else:

In [18]:
head_opt = tf.keras.optimizers.SGD(learning_rate=0.1)
before = [tf.identity(v) for v in finetune.weights]
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(finetune(X))
head_opt.apply_gradients(zip(tape.gradient(loss, finetune.trainable_weights),
                             finetune.trainable_weights))
[not bool(tf.reduce_all(b == v)) for b, v in zip(before, finetune.weights)]

[False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True]

Only the last two entries, the head's kernel and bias, changed. Two pitfalls
deserve attention here: one about optimizer memory, and one about batch
normalization, where Keras quietly protects you.

In [19]:
for layer in net.layers[:-1]:
    layer.trainable = False   # frozen, but Adam's moments remain allocated
moments = sum(int(tf.size(v)) for v in adam.variables if v.ndim > 0)
moments == 2 * n

True

Second, batch normalization, whose running statistics are non-trainable
weights recomputed by the forward pass in training mode. Elsewhere that is a
classic trap: freeze the layer's scale and shift and the statistics drift
anyway, since they never pass through the optimizer. Keras special-cases
exactly this: setting `trainable = False` on a `BatchNormalization` layer
also switches its forward pass to inference mode, so the statistics hold
still even when the layer is called with `training=True`:

In [20]:
bn = tf.keras.layers.BatchNormalization()
_ = bn(tf.zeros((8, 4)), training=True)   # build, then freeze
bn.trainable = False
before = tf.identity(bn.moving_mean)
_ = bn(tf.random.normal((8, 4)) + 3, training=True)
bool(tf.reduce_all(bn.moving_mean == before))

True

This courtesy is specific to `BatchNormalization`. Everywhere else,
`trainable` and the `training` call argument are orthogonal switches: the
first governs which variables receive updates, the second governs
training-mode behaviors, and a frozen dropout layer still drops when called
with `training=True`. To pin any other stateful layer during fine-tuning,
set both.

Freezing whole tensors is the bluntest form of partial training.
Parameter-efficient methods instead add small trainable low-rank corrections
next to frozen weights; the linear algebra behind them is developed in
that section. A related idea maintains derived,
non-trained state: an *exponential moving average* (EMA) of the weights kept
alongside the trained ones and used for evaluation. Whether it improves on the
final iterate depends on its decay and the training schedule; controlled
examples appear in that section. Like BatchNorm statistics,
the average is state updated outside backpropagation.

Keras builds the average into the optimizer. `Adam(use_ema=True)` maintains
an exponential moving average of every variable it updates, alongside its
moments, and `finalize_variable_values` copies the averages back into the
model, the usual last step before evaluation or saving. Continuing the
fine-tuning above, the swapped-in average sits measurably behind the last raw
iterate of the moving head:

In [21]:
ema_opt = tf.keras.optimizers.Adam(learning_rate=0.1, use_ema=True,
                                   ema_momentum=0.9)
for _ in range(5):
    with tf.GradientTape() as tape:
        loss = tf.reduce_sum(finetune(X))
    ema_opt.apply_gradients(zip(tape.gradient(loss, finetune.trainable_weights),
                                finetune.trainable_weights))
raw = tf.identity(finetune.layers[-1].bias)   # the last raw iterate
ema_opt.finalize_variable_values(finetune.trainable_weights)   # swap in averages
float(tf.reduce_max(tf.abs(finetune.layers[-1].bias - raw)))

0.3095071613788605

## Summary

A model's state is one list of variables, each named by the path of the
layers that own it. A per-variable flag splits the list into
`trainable_weights` and `non_trainable_weights`; weights created with
`add_weight(trainable=False)` persist and are saved but receive no updates,
and every variable's device is fixed at creation. Training with Adam in fp32
costs 16 bytes per parameter (4 weights, 4 gradients, 8 optimizer state)
before activations. Mixed-precision totals depend on the dtypes and copies the
implementation keeps. Tying is a head layer that owns no
variables and reuses the embedding table at a second call site: one entry in
`weights`, gradients summed over its uses. Setting `layer.trainable = False`
freezes a layer but reclaims no optimizer state already allocated;
`BatchNormalization` alone also stops updating its statistics when frozen,
and `Adam(use_ema=True)` keeps a weight average as part of the optimizer.

## Exercises

1. Write a helper that reports the byte cost of fp32 Adam training separately
   for each top-level block of `net`. Which block
   dominates, and would that still hold if the residual blocks were 10 times
   wider?
1. BatchNorm's running mean and variance are non-trained state. Suppose you
   made them trainable parameters so that the optimizer updates them. What
   goes wrong during training, and why does gradient descent on a running
   average not compute the same thing as the forward-pass update rule it
   replaces?

3. Round-trip the tied model's weights through `get_weights` and
   `set_weights` (or a checkpoint, that section) into a second
   tied instance, then into an untied one. Where, if anywhere, is the tying
   recorded? Explain why tying in Keras is a property of the layer code
   rather than of the checkpoint.
4. In the tied `TinyLM`, set `lm.emb.trainable = False` but leave `lm.head`
   alone. How many trainable parameters remain, and can the head still
   adapt? Explain what this teaches about the interaction between tying and
   freezing.